# 🧥 RTV: Real-Time Virtual Try-On Studio (Google Colab Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**RTV** is a real-time virtual try-on system powered by:
- **ROMP / BEV** for 3D body motion estimation
- **DensePose** for body-to-garment surface UV coordinates
- **PyOpenGL Offscreen Renderer** (Headless EGL on Linux cloud GPUs)
- **Pix2PixHD RGBA Generator** for realistic fabric texture and drape
- **Lucas–Kanade Optical Flow Tracker** for 25–30 FPS live motion tracking
- **Interactive Gradio Web Studio** supporting live webcam streaming, video upload, garment switcher, and telemetry HUD!

---
### ⚙️ Requirements:
- **Runtime**: Set Colab runtime to **GPU** (`Runtime` -> `Change runtime type` -> select `T4 GPU`, `L4 GPU`, or `A100 GPU`).

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM)")

## 2. Mount Google Drive & Set Up Project Repository
If your `RTV` code or checkpoints (`RTV.zip` or folder) are stored in your Google Drive, mount Drive below.

In [ ]:
import os

# Option A: Mount Google Drive if your RTV repository or RTV.zip is in Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted!")
except Exception as e:
    print("Drive mount skipped or running outside Colab:", e)

# Option B: If you uploaded RTV.zip to Drive (e.g. MyDrive/RTV.zip), uncomment below:
# if os.path.exists('/content/drive/MyDrive/RTV.zip'):
#     !unzip -q /content/drive/MyDrive/RTV.zip -d /content/
#     %cd /content/RTV

# Ensure working directory is set to RTV
if os.path.exists('/content/RTV'):
    os.chdir('/content/RTV')
print("Active Directory:", os.getcwd())

## 3. Install System Graphics & Headless EGL Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    libegl1-mesa-dev \
    libgles2-mesa-dev \
    libgl1-mesa-glx \
    libosmesa6-dev \
    xvfb \
    ffmpeg \
    libglfw3-dev > /dev/null 2>&1
print("✓ Headless graphics packages installed!")

## 4. Install Python Dependencies & Detectron2

In [ ]:
!pip install -q pyopengl pyopengl-accelerate trimesh simple-romp gradio==6.30.0 opencv-python scipy gdown
if os.path.exists('requirements.txt'):
    !pip install -q -r requirements.txt

# Install Detectron2 matching Colab PyTorch & CUDA
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print("✓ All Python dependencies installed successfully!")

## 5. Configure Headless OpenGL Environment

In [ ]:
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["QT_QPA_PLATFORM"] = "offscreen"
print("✓ Headless EGL and Offscreen platform configured!")

## 6. 🚀 Launch the Interactive Web Try-On Studio
Running the cell below launches the **RTV Interactive Web Studio**.

Features:
- **Live Webcam Try-On**: Stream your webcam directly from your browser to Colab with real-time cloth motion!
- **Video Try-On**: Upload any video or test the bundled sample clip.
- **Garment Selector**: Choose between Winter Puffer Coat, Hanfu Silk Dress, or tailored Upper-Body garments.
- **Telemetry HUD**: Displays real-time Cam FPS, ML latency, Tracking FPS, and body orientation.

> **Note:** Gradio will generate a public **`https://xxxx.gradio.live`** link. Click it to open the studio in a full browser tab or on your phone!

In [ ]:
# Launch Gradio with public shareable link
!python colab_gradio_app.py --share

## 7. Batch Process Video Headlessly (Optional CLI Mode)
If you want to render an entire video without keeping the Gradio web UI open, execute this cell:

In [1]:
from colab_gradio_app import ColabTryOnEngine
from IPython.display import Video, display

engine = ColabTryOnEngine()

input_video = "./assets/sample_video.mp4"
garment = "Winter Puffer Coat (Full-Body)"

print(f"Processing video: {input_video} with {garment}...")
out_video_path, msg = engine.process_video_file(
    video_path=input_video,
    garment_label=garment,
    max_frames=150,  # Number of frames to process
    show_hud=True
)
print(msg)
print(f"Resulting video saved to: {out_video_path}")

# Display output directly in notebook cell
display(Video(out_video_path, embed=True, width=640))

d:\projects\company-projects\RTV\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[RTV-Colab] Pre-warming FullBody model in GPU...
[RTV-Colab] Initializing FullBodySeqFrameProcessor (coat_seq_vmssdp2ta_576)...
Processing video: ./assets/sample_video.mp4 with Winter Puffer Coat (Full-Body)...
[RTV-Colab] Processing video: 150 frames at 15.0 FPS (640x426)
Using BEV.
Threshold for positive center detection: 0.08


: 